# State sync: Laminar to traitlets and back

The last piece. `own-bundle.ipynb` proved our Scala.js runs in the webview, but
its `render` ignored `model` on purpose. This one connects the two halves.

`Params` is defined **once**, in `example/src`, and compiled twice — into
`example.jvm` for the kernel and `example.js` for the browser. Neither end has a
hand-written JSON mapping. That is the argument for doing this in Scala at all.

```
./mill example.js.fullLinkJS
./mill example.jvm.publishLocal
```

In [ ]:
// Pulls wedgie-kernel in transitively.
import $ivy.`io.github.quafadas::wedgie-example:0.1.0-SNAPSHOT`

import wedgie.EsmSource
import wedgie.example.Params
import wedgie.kernel.Widget

## Load the bundle

Note the size. Adding the cross-compiled codec took this from 325 KB to ~1.4 MB:
`ModuleKind.ESModule` forfeits Closure, and upickle's derivation machinery is
exactly the code Closure was shrinking. `EsmSource.Inline` puts all of it in the
`comm_open` and in the saved `.ipynb`.

If that turns out to be too much, the one-line mitigation is:

```
npx esbuild out/example/js/fullLinkJS.dest/main.js --minify --format=esm \
  --outfile=out/example.min.js
```

which gets it to ~549 KB with the AFM still valid. `EsmSource.CommDelivered`
removes the `.ipynb` cost entirely, at the price of probes 4 and 5.

**Do not save this notebook** with the widget output in it.

In [ ]:
val bundlePath = java.nio.file.Paths.get(
  sys.props("user.home"), "Code", "wedgie",
  "out", "example", "js", "fullLinkJS.dest", "main.js"
)

val bundle = java.nio.file.Files.readString(bundlePath)
println(s"bundle: ${bundle.length / 1024} KB")

## The widget

Three controls, one shared state type. The initial value here is the kernel's;
the frontend reads every key off the model rather than using its own defaults.

In [ ]:
val params = Widget(Params(n = 25, label = "runs", enabled = true), EsmSource.Inline(bundle))

## Direction 1 — browser to kernel

Move the slider, type in the box, toggle the checkbox. Then run this.

No buffer to inspect and no manual decoding: the inbound `update` messages have
already been merged into a `Params`.

In [ ]:
params.state

## Direction 2 — kernel to browser

The controls should move without being touched. `set` diffs against current
state, so this puts only the changed keys on the wire.

In [ ]:
params.set(Params(n = 80, label = "pushed from Scala", enabled = false))

In [ ]:
// `modify` is the read-modify-write form.
params.modify(p => p.copy(n = math.min(100, p.n + 10)))

## The thing this is all for

A control driving a recalculation. Move the slider after running this, then
inspect the log.

`fromFrontend` matters: without it, an observer that calls `set` would drive
itself in a loop.

In [ ]:
val results = scala.collection.mutable.ArrayBuffer.empty[String]

def recompute(p: Params): String =
  if !p.enabled then s"${p.label}: disabled"
  else s"${p.label}: sum(1..${p.n}) = ${(1 to p.n).sum}"

params.onChange { change =>
  if change.fromFrontend then
    results.synchronized { results += recompute(change.state) }
}

In [ ]:
// Move the slider a few times first.
results.synchronized(results.toList).takeRight(10).foreach(println)

## No feedback loop

Both ends refuse to echo, by the same rule: diff against what the other side
already holds, and send nothing when the diff is empty.

- **Kernel**: an inbound `update` merges into state and is never sent back.
- **Frontend**: `Bridge` diffs against the model's *current* attributes, so a
  change that arrived from the kernel produces an empty diff and no `save_changes`.

Neither needs a mutex or a suppression flag. If a loop ever appears, this is the
invariant that broke.

Check it: the count below should not grow while you are not touching the widget.

In [ ]:
results.synchronized(results.size)

## What is left

- `Entry`/`smoke` stays as a bisection tool: it mounts Laminar without a model,
  so "Laminar broke" and "sync broke" remain separable.
- Bundle delivery is now the open question, and it is about size, not
  feasibility. See `probes.ipynb` if you want more than one widget per notebook.